## Export Genie Spaces

This notebook exports the approved Genie Spaces with their configurations, benchmarks, and permissions.

**What it does:**
1. Reads the approved inventory from Src_02
2. Exports each space's serialized_space configuration
3. Extracts and exports benchmarks
4. Exports permissions/ACLs
5. Writes consolidated CSVs for review

**Prerequisites:**
- CAN_EDIT permission on each Genie Space (required for serialized_space)
- Approved inventory exists from Src_02

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Setup Path Resolution

In [ ]:
import sys
import os

notebook_path = os.getcwd()
if notebook_path.startswith("/Workspace"):
    bundle_root = os.path.dirname(notebook_path)
else:
    bundle_root = os.path.dirname(os.path.abspath("__file__"))

helpers_path = os.path.join(bundle_root, "helpers")
if not os.path.exists(helpers_path):
    helpers_path = os.path.join(os.path.dirname(bundle_root), "helpers")

if helpers_path not in sys.path:
    sys.path.insert(0, os.path.dirname(helpers_path))

print(f"Bundle root: {bundle_root}")

## Read Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Export Volume Path")

volume_path = dbutils.widgets.get("volume_path")

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Export volume: {volume_path}")

## Load Approved Inventory

In [ ]:
import pandas as pd

approved_path = os.path.join(volume_path, "genie_inventory_approved", "inventory_approved.csv")

if not os.path.exists(approved_path):
    raise FileNotFoundError(
        f"Approved inventory not found: {approved_path}\n"
        "Run Src_02_Review_and_Approve first and confirm your selection."
    )

df = pd.read_csv(approved_path)
print(f"Loaded {len(df)} approved spaces")
display(df[["space_id", "title", "benchmark_count"]])

## Export Each Space

In [ ]:
from databricks.sdk import WorkspaceClient
from helpers.export import export_space, export_space_to_volume, write_export_manifest
from helpers.permissions import export_all_permissions
from helpers.benchmarks import export_all_benchmarks

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

exported_spaces = []
errors = []

for idx, row in df.iterrows():
    space_id = row["space_id"]
    title = row["title"]
    
    print(f"\n[{idx+1}/{len(df)}] Exporting: {title}")
    
    try:
        output_path = export_space_to_volume(w, space_id, volume_path)
        export_data = export_space(w, space_id, include_serialized=True)
        exported_spaces.append(export_data)
        print(f"  ✓ Exported to {output_path}")
    except Exception as e:
        error_msg = f"Failed to export {title}: {e}"
        print(f"  ✗ {error_msg}")
        errors.append({"space_id": space_id, "title": title, "error": str(e)})

print(f"\nExported {len(exported_spaces)} of {len(df)} spaces")

## Export Permissions

In [ ]:
print("\nExporting permissions...")

spaces_for_perms = [
    {"space_id": row["space_id"], "title": row["title"]}
    for _, row in df.iterrows()
]

try:
    perms_path = export_all_permissions(w, spaces_for_perms, volume_path)
    print(f"✓ Permissions exported to {perms_path}")
except Exception as e:
    print(f"✗ Failed to export permissions: {e}")

## Export Benchmarks

In [ ]:
print("\nExporting benchmarks...")

try:
    bm_path = export_all_benchmarks(exported_spaces, volume_path)
    print(f"✓ Benchmarks exported to {bm_path}")
except Exception as e:
    print(f"✗ Failed to export benchmarks: {e}")

## Write Export Manifest

In [ ]:
print("\nWriting export manifest...")

try:
    manifest_path = write_export_manifest(volume_path, exported_spaces)
    print(f"✓ Manifest written to {manifest_path}")
except Exception as e:
    print(f"✗ Failed to write manifest: {e}")

## Summary

In [ ]:
print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Spaces exported:   {len(exported_spaces)} / {len(df)}")
print(f"Export volume:     {volume_path}")
print(f"\nArtifacts written:")
print(f"  - exported/              (space JSON files)")
print(f"  - permissions/           (ACL exports)")
print(f"  - benchmarks/            (benchmark exports)")
print(f"  - export_manifest.json   (summary)")

if errors:
    print(f"\n⚠️  Errors ({len(errors)}):")
    for err in errors:
        print(f"  - {err['title']}: {err['error']}")

print("\nNext step: Deploy target bundle and run tgt_genie_deploy job.")